# Guided Project: Building From Rank One
**MATH 3515 · Module 14 · First segment of the Challenge Project *Keeping What Matters***

**The story.** Your phone holds thousands of photos. Each one is a grid of numbers: a grayscale photo with 512 rows and 384 columns is a matrix with almost 200,000 entries. **How few numbers can describe a photo well enough?** The SVD answers this: it writes the photo as a sum of simple rank-one pieces $\sigma_i\mathbf u_i\mathbf v_i^T$, ordered from most to least important, and keeping the first $r$ pieces gives the best possible approximation with that many pieces.

The same tool answers a second, older question: **how do you fit a line (or a curve) to data that no line passes through exactly?** That is **least squares**. The usual textbook recipe, the normal equations, squares the condition number and can lose half the digits; the SVD does not.

In this Guided Project you **learn least squares and collect the evidence**: on a small example by hand, on polynomial fits, and on two of your own photos. Explaining the results, choosing how many pieces each photo needs, and the color analysis are the Challenge Project *Keeping What Matters*.

**What you hand in** (due date in Blackboard): only the report `GP_Building_From_Rank_One_Report.docx`, with your two **clean** photos inserted. **This notebook is not collected:** keep it (and your two photos) for the Challenge Project. Write short *Observations* where asked (1–2 sentences: what you see, not why).

**Resources:** the Module 12 slides on the SVD and Cholesky; class notebook 07 (*Singular Values, the SVD, and Cholesky*); Problem Set 2.
**AI use: Level 2 (AI Collaboration).** You may use an assistant for code and for checking your reasoning; you are responsible for verifying everything against the notes, your hand work, and this notebook.

**What you type:** only the file names of your two photos (Part 0). Everything else is given: run it, look at the output, and write a short *Observation* where asked. Part 1 is done on paper first. Part 6 belongs to the Challenge Project; skip it now.

In [ ]:
# Setup: run this cell first
import io, os, urllib.request
import numpy as np
import scipy.linalg as sl
import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError
np.set_printoptions(precision=4, suppress=True)

# short error messages: show only the last line of an error
try:
    get_ipython().run_line_magic("xmode", "Minimal")
except NameError:
    pass

# Colab tools for uploading and downloading files (the notebook also runs outside Colab)
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Okabe-Ito colorblind-safe palette
OI = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#000000"]

SAMPLES = {"pet": "sample_dog.jpg", "landscape": "sample_windmill.jpg"}
SAMPLE_URL = "https://raw.githubusercontent.com/acintron-cc/math3515-202710/main/notebooks/data/"
HEIC_MSG = ("This looks like an iPhone HEIC photo. Export it as JPEG "
            "(or set Settings › Camera › Formats › Most Compatible) and upload again.")


def size_text(rows, cols):
    return f"{rows} rows x {cols} columns"


def find_photo(name, kind):
    # turn what you typed into a file path; every problem stops with a one-line message
    name = name.strip()
    if name in ("", "___"):
        raise ValueError('Type the file names of your two photos (or "sample").')
    if name.lower() == "sample":
        local = os.path.join("data", SAMPLES[kind])      # a local copy of the course repository
        if os.path.exists(local):
            return local
        if not os.path.exists(SAMPLES[kind]):
            try:
                with urllib.request.urlopen(SAMPLE_URL + SAMPLES[kind], timeout=10) as f:
                    data = f.read()
            except Exception:
                raise ConnectionError("Could not download the sample photo. Check the internet connection and run the cell again.") from None
            with open(SAMPLES[kind], "wb") as f:
                f.write(data)
        return SAMPLES[kind]
    if name.lower().endswith((".heic", ".heif")):
        raise ValueError(HEIC_MSG)
    if not os.path.exists(name):
        here = sorted(f for f in os.listdir(".") if os.path.isfile(f) and not f.startswith("."))
        listing = ", ".join(here[:20]) + (", ..." if len(here) > 20 else "") if here else "(none)"
        raise FileNotFoundError(f"File not found: {name}. Files here: {listing}. Upload it with the cell above and check the name.")
    return name


def load_photo(path):
    # read only the pixels, crop uniform borders, shrink to at most 512 pixels a side
    try:
        im = Image.open(path)
        im.load()
    except UnidentifiedImageError:
        with open(path, "rb") as f:
            head = f.read(32)
        if b"ftyphei" in head or b"ftyphev" in head or b"ftypmif1" in head:
            raise ValueError(HEIC_MSG) from None
        raise ValueError(f"{os.path.basename(path)} is not a photo this notebook can read. Upload a JPEG or PNG.") from None
    rows0, cols0 = im.height, im.width
    im = im.convert("RGB")                                  # drops transparency and palettes
    gray = np.asarray(im, dtype=float).mean(axis=2)         # 0..255
    keep_r = np.where(gray.std(axis=1) >= 1.0)[0]           # rows that are not one flat color
    keep_c = np.where(gray.std(axis=0) >= 1.0)[0]
    if keep_r.size == 0 or keep_c.size == 0:
        raise ValueError(f"{os.path.basename(path)} is one flat color. Choose a real photo.")
    im = im.crop((keep_c[0], keep_r[0], keep_c[-1] + 1, keep_r[-1] + 1))
    rows1, cols1 = im.height, im.width
    im.thumbnail((512, 512), Image.LANCZOS)
    X = np.asarray(im, dtype=float) / 255                    # color, m x n x 3
    A = X.mean(axis=2)                                       # grayscale: mean of the three channels
    lam = np.linalg.eigvalsh(np.cov(X.reshape(-1, 3).T))     # color variance along its three main directions
    gray_share = lam.max() / lam.sum() if lam.sum() > 0 else 1.0
    return {"file": os.path.basename(path), "image": im, "X": X, "A": A,
            "original": (rows0, cols0), "cropped": (rows1, cols1), "gray_share": gray_share}

print("Ready.")

## Part 0. Your two photos  ✎ YOUR TURN
You need **two of your own color photos**: one of a **pet or an object**, one **landscape** (nature, or a place from a recent trip).
- **No people anywhere, including the background.** Statues are fine.
- Your own photos, not downloaded ones; original photos, no frames or filters; JPEG or PNG.
- Nothing that identifies your home (house numbers, street signs, license plates, pet tags with phone numbers).

**Steps.** (1) Type the two file names in the next cell (or `"sample"` for both, to try the notebook with the instructor's photos). (2) Run the upload cell: in Colab it opens a file chooser for the photos that are not here yet, and prints the exact names it received (Colab renames a file uploaded twice, e.g. `IMG_1234 (1).jpg`: copy that name into the first cell). (3) Run the third cell.

The notebook reads **only the pixels**: phone photos can carry the GPS location, and the clean copies `clean_pet.jpg` and `clean_landscape.jpg` it saves have none. **Insert the clean copies in your report**, not the originals. Colab deletes uploaded files when the session ends, so keep your photos.

In [ ]:
PET_PHOTO = "___"         # pet or object: the file name in quotes, e.g. "IMG_1234.jpg" (or "sample")
LANDSCAPE_PHOTO = "___"   # landscape: the file name in quotes (or "sample")

In [ ]:
# Upload helper (given): opens a file chooser only for photos you named that are not here yet
needed = [p.strip() for p in (PET_PHOTO, LANDSCAPE_PHOTO)
          if p.strip().lower() not in ("", "___", "sample") and not os.path.exists(p.strip())]
if not needed:
    print("Nothing to upload.")
elif IN_COLAB:
    print("Choose these photos in the file chooser:", ", ".join(needed))
    uploaded = files.upload()
    print("Received these files (use exactly these names in the cell above):")
    for received in uploaded:
        print("   " + received)
else:
    print("Not in Colab: copy these photos into the folder of this notebook:", ", ".join(needed))

In [ ]:
# Load both photos (given): crop uniform borders, shrink, grayscale, save clean copies
photos = {}
for kind, typed in (("pet", PET_PHOTO), ("landscape", LANDSCAPE_PHOTO)):
    P = load_photo(find_photo(typed, kind))
    photos[kind] = P
    m, n = P["A"].shape
    (r0, c0), (r1, c1) = P["original"], P["cropped"]
    print(f"{kind} photo: {P['file']}")
    print(f"   original size       {size_text(r0, c0)}")
    print(f"   after cropping      {size_text(r1, c1)}  ({r0 - r1} border rows and {c0 - c1} border columns removed)")
    print(f"   final size          {size_text(m, n)}  (m = {m}, n = {n})")
    if P["gray_share"] >= 0.995:
        print("   Your photo is (almost) black and white. That is fine here, but the Challenge Project's color analysis needs a color photo.")
    clean = f"clean_{kind}.jpg"
    Image.fromarray(np.asarray(P["image"])).save(clean, quality=95)   # a new image from the pixels only
    print(f"   clean copy saved:   {clean} (no camera or location data)")
    if IN_COLAB and typed.strip().lower() != "sample":
        files.download(clean)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, kind in zip(axes, ("pet", "landscape")):
    ax.imshow(photos[kind]["X"])
    ax.set_title(f"{kind}: clean copy, {size_text(*photos[kind]['A'].shape)}")
    ax.axis("off")
plt.tight_layout()
plt.show()
print("Check: no people anywhere (also in the background), and nothing that identifies your home "
      "(house numbers, street signs, license plates, pet tags with phone numbers).")

**Check: no people anywhere (also in the background), and nothing that identifies your home (house numbers, street signs, license plates, pet tags with phone numbers).** If a photo breaks a rule, choose another one and run Part 0 again.

## Part 1. Least squares by hand (on paper first)
Four data points: $(0,1),\ (1,3),\ (1,1),\ (4,5)$. We want a line $y\approx c_0+c_1x$.

**No line passes through all four points** (two of them even have the same $x$). **Least squares** chooses the line that makes the **sum of squared vertical misses** (the residuals $r_i=y_i-(c_0+c_1x_i)$) as small as possible. In matrix form, with one row per point:
$$A=\begin{pmatrix}1&0\\1&1\\1&1\\1&4\end{pmatrix},\qquad \mathbf y=\begin{pmatrix}1\\3\\1\\5\end{pmatrix},\qquad \mathbf c=\begin{pmatrix}c_0\\c_1\end{pmatrix},\qquad \text{minimize } \|A\mathbf c-\mathbf y\|_2 .$$
At the best $\mathbf c$, the residual vector $\mathbf y-A\mathbf c$ is perpendicular to both columns of $A$: $A^T(\mathbf y-A\mathbf c)=\mathbf 0$. That gives the **normal equations**
$$A^TA\,\mathbf c=A^T\mathbf y .$$
$A^TA$ is symmetric positive definite here, so it has a **Cholesky factorization** $A^TA=LL^T$ (Module 12).

**Tasks: by hand, before running the next cell** (handwritten or typed; show every step):
- **(a)** Compute $A^TA$ and $A^T\mathbf y$. Say what each entry is: $4$ = number of points, $6=\sum x_i$, $18=\sum x_i^2$, $10=\sum y_i$, $24=\sum x_iy_i$.
- **(b)** Cholesky: $A^TA=LL^T$ with $L$ lower triangular.
- **(c)** Forward solve $L\mathbf z=A^T\mathbf y$, then back solve $L^T\mathbf c=\mathbf z$. Write the line.
- **(d)** The four residuals $r_i=y_i-(c_0+c_1x_i)$ and the sum of squared residuals.

**What does it tell you?** (1–2 sentences each)
- **(e)** Add the residuals; then add $x_i\,r_i$. Which of the two normal equations does each sum come from?
- **(f)** The two points at $x=1$ have $y=3$ and $y=1$. What does the line do with them?
- **(g)** Check that the line passes through the average point $(\bar x,\bar y)$. Which normal equation guarantees it?

Then run the cell and compare its values with yours in the report's **Handwritten Computations** table.

In [ ]:
# Check your hand work (given)
x = np.array([0.0, 1.0, 1.0, 4.0])
y = np.array([1.0, 3.0, 1.0, 5.0])
A1 = np.column_stack([np.ones(4), x])           # columns: ones, x

AtA = A1.T @ A1
Aty = A1.T @ y
L = np.linalg.cholesky(AtA)                       # AtA = L L^T
z = sl.solve_triangular(L, Aty, lower=True)       # forward solve  L z = A^T y
c = sl.solve_triangular(L.T, z, lower=False)      # back solve     L^T c = z
res = y - A1 @ c

print("A^T A =\n", AtA)
print("A^T y =", Aty)
print("L =\n", L)
print("z =", z)
print(f"c = {c}    line: y = {c[0]:.4g} + {c[1]:.4g} x")
print("residuals r =", res)
print(f"sum of squared residuals = {np.sum(res**2):.4g}")
print(f"sum of r_i = {np.sum(res):.4g},   sum of x_i r_i = {np.sum(x * res):.4g}")
print(f"average point (x_bar, y_bar) = ({x.mean():.4g}, {y.mean():.4g});   line at x_bar: {c[0] + c[1] * x.mean():.4g}")

fig, ax = plt.subplots(figsize=(6, 4))
xx = np.linspace(-0.3, 4.3, 50)
ax.plot(xx, c[0] + c[1] * xx, "-", color=OI[0], label=f"least-squares line y = {c[0]:.3g} + {c[1]:.3g} x")
ax.vlines(x, y, A1 @ c, colors=OI[1], linestyles="--", label="residuals (vertical misses)")
ax.plot(x, y, "o", color=OI[6], markersize=7, label="data points")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("Part 1: the least-squares line and its residuals")
ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.show()

## Part 2. Why not always use the normal equations? (run and observe)
**(a)** The condition number of the Part 1 matrix, and of $A^TA$.

In [ ]:
kappa_A = np.linalg.cond(A1)            # 2-norm condition number: sigma_max / sigma_min
kappa_AtA = np.linalg.cond(AtA)
print(f"kappa_2(A)      = {kappa_A:.4f}")
print(f"kappa_2(A^T A)  = {kappa_AtA:.4f}")
print(f"kappa_2(A)^2    = {kappa_A**2:.4f}")

Forming $A^TA$ **squares the condition number**: $\kappa_2(A^TA)=\kappa_2(A)^2$. Since the digits lost are about $\log_{10}\kappa$ (Module 12), the normal equations can lose **twice as many digits** as a method that works with $A$ itself. Here $\kappa$ is small and nothing is lost; the next experiment makes $\kappa$ large.

**(b) Polynomial fits with a known answer.** Take 50 points $x$ between 0 and 1 and the data $y=1+x+x^2+\dots+x^d$, **without noise**, so the exact coefficients are all 1 and every error is numerical. For each degree $d=1,\dots,12$, the matrix $A$ has columns $1, x, \dots, x^d$. We solve the least-squares problem two ways:
1. **normal equations with Cholesky** (form $A^TA$, factor, two triangular solves);
2. **`np.linalg.lstsq`**, which uses the SVD of $A$ and never forms $A^TA$.

For each, the relative error $\|\hat{\mathbf c}-\mathbf 1\|_2/\|\mathbf 1\|_2$ and the **correct digits** $=-\log_{10}(\text{error})$ (at most 16).

In [ ]:
# Given: the experiment (the plot and the table follow)
xs = np.linspace(0, 1, 50)

def correct_digits(err):
    return 16.0 if err == 0 else min(16.0, -np.log10(err))

poly = []          # one row per degree: d, kappa_2(A), kappa_2(A^T A), error normal eq. (None if Cholesky fails), error lstsq
for d in range(1, 13):
    V = np.vander(xs, d + 1, increasing=True)         # columns 1, x, ..., x^d
    c_true = np.ones(d + 1)
    yd = V @ c_true
    try:
        Lv = np.linalg.cholesky(V.T @ V)
        c_ne = sl.solve_triangular(Lv.T, sl.solve_triangular(Lv, V.T @ yd, lower=True), lower=False)
        err_ne = np.linalg.norm(c_ne - c_true) / np.linalg.norm(c_true)
    except np.linalg.LinAlgError:
        err_ne = None                                  # Cholesky fails
    c_ls = np.linalg.lstsq(V, yd, rcond=None)[0]
    err_ls = np.linalg.norm(c_ls - c_true) / np.linalg.norm(c_true)
    poly.append((d, np.linalg.cond(V), np.linalg.cond(V.T @ V), err_ne, err_ls))
print("done:", len(poly), "degrees")

In [ ]:
# Plot first (given): correct digits and condition numbers against the degree
ds = [p[0] for p in poly]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot([p[0] for p in poly if p[3] is not None], [correct_digits(p[3]) for p in poly if p[3] is not None],
         "o-", color=OI[1], label="normal equations + Cholesky")
ax1.plot(ds, [correct_digits(p[4]) for p in poly], "s--", color=OI[0], label="lstsq (SVD)")
ax1.axhline(0, color=OI[6], linewidth=0.8)
ax1.set_xlabel("degree d"); ax1.set_ylabel("correct digits")
ax1.set_title("Correct digits of the coefficients")
ax1.grid(alpha=0.3); ax1.legend(fontsize=8)
ax2.plot(ds, [np.log10(p[1]) for p in poly], "^-", color=OI[2], label=r"$\log_{10}\kappa_2(A)$")
ax2.plot(ds, [np.log10(p[2]) for p in poly], "v:", color=OI[3], label=r"$\log_{10}\kappa_2(A^TA)$")
ax2.set_xlabel("degree d"); ax2.set_ylabel("log10 of the condition number")
ax2.set_title("Condition numbers")
ax2.grid(alpha=0.3); ax2.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Table (given)
def digits_text(err):
    if err is None:
        return "Cholesky fails"
    dg = correct_digits(err)
    return f"{dg:.1f}" + (" (none)" if dg < 0 else "")

print(f"{'d':>3} {'kappa_2(A)':>11} {'kappa_2(AtA)':>13} {'normal eq. error':>17} {'digits':>12} {'lstsq error':>12} {'digits':>7}")
for d, kA, kG, en, el in poly:
    en_text = "-" if en is None else f"{en:.1e}"
    print(f"{d:>3} {kA:>11.2e} {kG:>13.2e} {en_text:>17} {digits_text(en):>12} {el:>12.1e} {correct_digits(el):>7.1f}")

**Observation.** How do the correct digits of the two methods change as $d$ grows? Compare with $\log_{10}\kappa_2(A)$ and $\log_{10}\kappa_2(A^TA)$.

*Your observation:*

## Part 3. The pseudoinverse (run and observe)
With the SVD $A=U\Sigma V^T$ (economy size), the least-squares solution is
$$\mathbf x=V\Sigma^+U^T\mathbf y,\qquad \Sigma^+=\operatorname{diag}(1/\sigma_1,\dots,1/\sigma_k,0,\dots,0),$$
where only the singular values above a small **tolerance** are inverted; the others count as zero. $A^+=V\Sigma^+U^T$ is the **pseudoinverse**. The SVD is done only by computer.

**(a)** The Part 1 data again.

In [ ]:
U1, s1, Vt1 = np.linalg.svd(A1, full_matrices=False)
x_pinv = Vt1.T @ ((U1.T @ y) / s1)
print("singular values of A:", s1)
print("x = V Sigma^+ U^T y =", x_pinv, "   (Part 1: c =", c, ")")

**(b) A rank-deficient case.** Add a third column $2x$: the same information twice. $A_3=[\,\mathbf 1,\ x,\ 2x\,]$. Now $A_3^TA_3$ is singular, Cholesky cannot work, and there are infinitely many best coefficient vectors.

In [ ]:
A3 = np.column_stack([np.ones(4), x, 2 * x])
AtA3 = A3.T @ A3
print("A3^T A3 =\n", AtA3)
print(f"det = {np.linalg.det(AtA3):.1e},  rank of A3 = {np.linalg.matrix_rank(A3)}")
try:
    np.linalg.cholesky(AtA3)
    chol3 = "Cholesky succeeds"
except np.linalg.LinAlgError as err:
    chol3 = f"Cholesky fails ({err})"
print(chol3)

U3, s3, Vt3 = np.linalg.svd(A3, full_matrices=False)
tol = max(A3.shape) * np.finfo(float).eps * s3[0]
s3_plus = np.array([1 / v if v > tol else 0.0 for v in s3])     # invert only the singular values above tol
x3 = Vt3.T @ (s3_plus * (U3.T @ y))
print(f"singular values: {s3[0]:.4f}, {s3[1]:.4f}, {s3[2]:.1e}   tolerance = {tol:.1e}  (sigma_3 counts as zero)")
print("pseudoinverse x  =", x3)
print("np.linalg.lstsq  =", np.linalg.lstsq(A3, y, rcond=None)[0])
print("np.linalg.pinv   =", np.linalg.pinv(A3) @ y)
print("fitted values A3 x =", A3 @ x3, "   Part 1:", A1 @ c)
print("residuals          =", y - A3 @ x3)
print(f"c1 + 2 c2 = {x3[1] + 2 * x3[2]:.4g}  (the slope of the Part 1 line)")
print(f"length ||x|| = {np.linalg.norm(x3):.4f}   vs  ||(1, 1, 0)|| = {np.linalg.norm([1, 1, 0]):.4f}")

**Observation.** What did Cholesky do with $A_3^TA_3$, and what did the pseudoinverse give? Compare its fitted values with Part 1.

*Your observation:*

## Part 4. Your photos with fewer pieces (run and observe)
Each grayscale photo is an $m\times n$ matrix $A$ with entries between 0 (black) and 1 (white). Its SVD writes it as a sum of rank-one pieces, and the **rank-$r$ version** keeps the first $r$:
$$A_r=\sigma_1\mathbf u_1\mathbf v_1^T+\dots+\sigma_r\mathbf u_r\mathbf v_r^T,\qquad \|A-A_r\|_2=\sigma_{r+1}.$$
- **Relative error** in the Frobenius norm, computed two ways that must agree: directly, $\|A-A_r\|_F/\|A\|_F$, and from the singular values, $\sqrt{\sigma_{r+1}^2+\dots}\,/\sqrt{\sigma_1^2+\dots}$.
- **Storage** of $A_r$: $r$ columns $\mathbf u_i$ ($m$ numbers each), $r$ rows $\mathbf v_i^T$ ($n$ each) and $r$ numbers $\sigma_i$, so $r(m+n+1)$ numbers instead of $mn$.

In [ ]:
# Given: one SVD per photo (about 0.1 s each)
for kind in ("pet", "landscape"):
    P = photos[kind]
    U, s, Vt = np.linalg.svd(P["A"], full_matrices=False)
    P["U"], P["s"], P["Vt"] = U, s, Vt
    tail = np.sqrt(np.cumsum((s**2)[::-1])[::-1] / np.sum(s**2))    # tail[r] = relative error of rank r, r = 0..p-1
    P["rel_from_sigma"] = np.append(tail, 0.0)                        # rank p (all pieces): error 0
    m, n = P["A"].shape
    print(f"{kind}: {size_text(m, n)},  sigma_1 = {s[0]:.3f},  smallest sigma = {s[-1]:.3e}")

def rank_r(P, r):
    return (P["U"][:, :r] * P["s"][:r]) @ P["Vt"][:r, :]

def storage(P, r):
    m, n = P["A"].shape
    return 100 * r * (m + n + 1) / (m * n)          # percent of m*n

def smallest_r(P, target):
    return int(np.argmax(P["rel_from_sigma"][1:] <= target)) + 1

fig, ax = plt.subplots(figsize=(7, 4))
for kind, style, color in (("pet", "-", OI[1]), ("landscape", "--", OI[0])):
    s = photos[kind]["s"]
    ax.semilogy(np.arange(1, len(s) + 1), np.maximum(s, 1e-16 * s[0]), style, color=color, linewidth=2,
                label=f"{kind} ({size_text(*photos[kind]['A'].shape)})")
ax.set_ylim(bottom=max(1e-16 * max(photos[k]["s"][0] for k in photos), 0.5 * min(photos[k]["s"][-1] for k in photos)))
ax.set_xlabel("index k"); ax.set_ylabel("sigma_k (log scale)")
ax.set_title("Singular values of your two photos")
ax.grid(alpha=0.3, which="both"); ax.legend()
plt.show()

Rank-$r$ versions side by side with the original, for $r=5, 20, 50$.

In [ ]:
# Given: rank-r images
ranks_shown = [5, 20, 50]
fig, axes = plt.subplots(2, len(ranks_shown) + 1, figsize=(14, 7.5))
for row, kind in enumerate(("pet", "landscape")):
    P = photos[kind]
    m, n = P["A"].shape
    axes[row, 0].imshow(P["A"], cmap="gray", vmin=0, vmax=1)
    axes[row, 0].set_title(f"{kind}: original\n{size_text(m, n)}, rank {len(P['s'])}", fontsize=10)
    for ax, r in zip(axes[row, 1:], ranks_shown):
        if r >= len(P["s"]):
            ax.set_title(f"rank {r}: more than the photo has", fontsize=10)
        else:
            ax.imshow(np.clip(rank_r(P, r), 0, 1), cmap="gray", vmin=0, vmax=1)
            ax.set_title(f"rank r = {r}\nerror {100 * P['rel_from_sigma'][r]:.1f}%, storage {storage(P, r):.1f}%", fontsize=10)
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Given: the table for each photo
for kind in ("pet", "landscape"):
    P = photos[kind]
    A, s = P["A"], P["s"]
    m, n = A.shape
    print(f"{kind} ({P['file']}): {size_text(m, n)},  sigma_1 = {s[0]:.3f}")
    print(f"{'r':>5} {'rel. error (direct)':>20} {'(from sigmas)':>14} {'||A - A_r||_2':>14} {'sigma_(r+1)':>12} {'storage':>8}")
    for r in (1, 5, 20, 50, 100):
        if r >= len(s):
            continue
        E = A - rank_r(P, r)
        direct = np.linalg.norm(E) / np.linalg.norm(A)
        print(f"{r:>5} {direct:>20.4f} {P['rel_from_sigma'][r]:>14.4f} {np.linalg.norm(E, 2):>14.4f} {s[r]:>12.4f} {storage(P, r):>7.1f}%")
    for target in (0.10, 0.05, 0.02):
        r = smallest_r(P, target)
        print(f"   smallest r for {100 * target:.0f}% relative error: r = {r:>3},  storage {storage(P, r):.1f}% of m*n")
    print()

Compare your two photos. Which one needs more pieces? Can a "compressed" version need more storage than the photo itself?

**Observation.** (What you see, 1–2 sentences.)

*Your observation:*

## Part 5. Your results for the report (given)
Copy these two tables into the report's **1. Results** section (they are Markdown; they also paste into Word as text you can convert to a table).

In [ ]:
by_d = {p[0]: p for p in poly}
print("Table 1. Least squares")
print(f"Part 1: c = ({c[0]:.4g}, {c[1]:.4g}), line y = {c[0]:.4g} + {c[1]:.4g} x;  kappa_2(A) = {kappa_A:.4f},  kappa_2(A^T A) = {kappa_AtA:.4f}\n")
print("| d | kappa_2(A) | correct digits, normal eq. | correct digits, lstsq |")
print("|---|---|---|---|")
for d in (3, 6, 9, 12):
    _, kA, _, en, el = by_d[d]
    print(f"| {d} | {kA:.2e} | {digits_text(en)} | {correct_digits(el):.1f} |")
print(f"\nPart 3(b), rank-deficient A3: {chol3}; pseudoinverse x = ({x3[0]:.4g}, {x3[1]:.4g}, {x3[2]:.4g})\n")

print("Table 2. My photos")
print("| photo | size (rows x columns) | r for 10% | storage | r for 5% | storage | r for 2% | storage |")
print("|---|---|---|---|---|---|---|---|")
for kind in ("pet", "landscape"):
    P = photos[kind]
    m, n = P["A"].shape
    cells = []
    for target in (0.10, 0.05, 0.02):
        r = smallest_r(P, target)
        cells += [str(r), f"{storage(P, r):.1f}%"]
    print(f"| {kind} ({P['file']}) | {size_text(m, n)} | " + " | ".join(cells) + " |")
print("\nInsert your two clean photos (clean_pet.jpg, clean_landscape.jpg) in the report.")

## Part 6 (Challenge Project, skip now)
In the Challenge Project *Keeping What Matters* you will choose how many pieces each photo needs and defend the choice, study the photos' **colors** (each pixel is a point (R, G, B): principal component analysis), and compare three "best lines" through the pixel cloud of two colors you choose: least squares one way, least squares the other way, and the first principal component. Keep this notebook and your two photos.

## Before you submit the report
- Run **Kernel → Restart & Run All** (Colab: *Runtime → Restart and run all*) so every output is fresh, then save.
- Check that the numbers in your report match this notebook.
- Check that the two **clean** photos (`clean_pet.jpg`, `clean_landscape.jpg`) are inserted in the report.
- Submit your Guided Project report (`.docx`). Keep this notebook and your photos: the Challenge Project builds on them.